<a href="https://colab.research.google.com/github/al7arbi-111/honest-backtester/blob/main/honest_backtester.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np, pandas as pd, yfinance as yf

class Backtest:
    def __init__(self, prices, cost_bps=5, train_end="2023-12-31"):
        self.prices = prices
        self.returns = prices.pct_change().dropna()
        self.cost = cost_bps / 10000
        self.train_end = train_end

    def train(self):
        return self.returns.loc[:self.train_end]

    def test(self):
        return self.returns.loc[self.train_end:].iloc[1:]

In [7]:
def run(self, strategy, data=None):
    data = self.test() if data is None else data
    if isinstance(data, pd.DataFrame) and data.shape[1] == 1:
        data = data.iloc[:, 0]
    raw = pd.Series([float(strategy(data.iloc[:i])) for i in range(1, len(data)+1)],
                    index=data.index, dtype=float)
    pos = raw.shift(1).fillna(0)
    gross = pos * data
    fees = pos.diff().abs().fillna(0) * self.cost
    return gross - fees

Backtest.run = run

In [3]:
def stats(self, net, rf=0.04):
    n = len(net)
    ann_r = net.mean() * 252
    ann_v = net.std() * 252 ** 0.5
    sharpe = (ann_r - rf) / ann_v if ann_v > 0 else np.nan
    eq = (1 + net).cumprod()
    active = net[net != 0]
    t_stat = sharpe * np.sqrt(n / 252) if ann_v > 0 else np.nan
    return {
        "days": n,
        "annual_return": ann_r,
        "volatility": ann_v,
        "sharpe": sharpe,
        "max_drawdown": (eq / eq.cummax() - 1).min(),
        "win_rate": (active > 0).mean() if len(active) else np.nan,
        "t_stat": t_stat,
        "significant": abs(t_stat) > 1.96 if not np.isnan(t_stat) else False,
    }

Backtest.stats = stats

In [9]:
def report(self, strategy, name="strategy"):
    net = self.run(strategy)
    s = self.stats(net)
    print(f"=== {name} ===")
    print(f"  period        {s['days']} days")
    print(f"  annual ret    {s['annual_return']*100:.1f}%")
    print(f"  volatility    {s['volatility']*100:.1f}%")
    print(f"  sharpe        {s['sharpe']:.2f}")
    print(f"  max drawdown  {s['max_drawdown']*100:.1f}%")
    print(f"  win rate      {s['win_rate']*100:.1f}%")
    print(f"  t-statistic   {s['t_stat']:.2f}")
    print(f"  verdict       {'significant' if s['significant'] else 'NOT distinguishable from zero'}")
    return None

Backtest.report = report

In [8]:
def momentum(data):
    if len(data) < 50:
        return 0
    return 1 if data.iloc[-50:].mean() > 0 else -1

def buy_and_hold(data):
    return 1

bt = Backtest(prices, cost_bps=5)
bt.report(momentum, "50-day momentum")
print()
bt.report(buy_and_hold, "buy and hold")

=== 50-day momentum ===
  period        501 days
  annual ret    11.0%
  volatility    16.0%
  sharpe        0.44
  max drawdown  -18.0%
  win rate      56.1%
  t-statistic   0.62
  verdict       NOT distinguishable from zero

=== buy and hold ===
  period        501 days
  annual ret    21.4%
  volatility    16.4%
  sharpe        1.06
  max drawdown  -18.8%
  win rate      58.2%
  t-statistic   1.50
  verdict       NOT distinguishable from zero


{'days': 501,
 'annual_return': np.float64(0.21388050497738662),
 'volatility': 0.163659096677139,
 'sharpe': np.float64(1.062455485260389),
 'max_drawdown': -0.18755234161457868,
 'win_rate': np.float64(0.582),
 't_stat': np.float64(1.4980604401338633),
 'significant': np.False_}

In [10]:
def compare(self, strategies):
    rows = []
    for name, fn in strategies.items():
        s = self.stats(self.run(fn))
        rows.append({"strategy": name, "return": s["annual_return"]*100,
                     "vol": s["volatility"]*100, "sharpe": s["sharpe"],
                     "t_stat": s["t_stat"], "significant": s["significant"]})
    df = pd.DataFrame(rows).set_index("strategy").round(2)
    return df

Backtest.compare = compare

bt.compare({"50-day momentum": momentum, "buy and hold": buy_and_hold})

,return,vol,sharpe,t_stat,significant
strategy,,,,,
50-day momentum,11.01,16.00,0.44,0.62,False
buy and hold,21.39,16.37,1.06,1.50,False


In [11]:
def test_engine():
    idx = pd.date_range("2024-01-01", periods=100, freq="B")
    flat = pd.Series(100.0, index=idx)
    up = pd.Series(np.arange(100, 200.0), index=idx)

    b = Backtest(pd.DataFrame({"X": flat}), cost_bps=0)
    assert abs(b.run(lambda d: 1).sum()) < 1e-9, "flat prices must give zero return"

    b = Backtest(pd.DataFrame({"X": up}), cost_bps=0, train_end="2024-02-01")
    long_r = b.run(lambda d: 1).sum()
    short_r = b.run(lambda d: -1).sum()
    assert long_r > 0, "rising prices must profit a long"
    assert abs(long_r + short_r) < 1e-9, "long and short must be exact mirrors"

    b_free = Backtest(pd.DataFrame({"X": up}), cost_bps=0, train_end="2024-02-01")
    b_paid = Backtest(pd.DataFrame({"X": up}), cost_bps=10, train_end="2024-02-01")
    flip = lambda d: 1 if len(d) % 2 else -1
    assert b_paid.run(flip).sum() < b_free.run(flip).sum(), "costs must reduce returns"

    cheat = lambda d: 1
    r = b.run(cheat)
    assert r.iloc[0] == 0, "first day must be flat — position lags by one day"

    print("all tests passed")

test_engine()

all tests passed
